# DynamoDB works-graph Table Restore

Rollback plan for restoring the `catalogue-2025-10-02_works-graph` DynamoDB table from a PITR-restored copy.

**Steps:**
1. Clear all items from the corrupted `catalogue-2025-10-02_works-graph` table
2. Scan `restored-catalogue-2025-10-02_works-graph` and batch-write every item into the original table
3. Verify item counts match

**Prerequisites:**
- The PITR restore into `restored-catalogue-2025-10-02_works-graph` must already be complete
- Active AWS SSO session: `aws sso login`

In [1]:
import os
import time
import random
from concurrent.futures import ThreadPoolExecutor, as_completed

import boto3
from botocore.config import Config

os.environ.setdefault("AWS_PROFILE", "platform-developer")

AWS_PROFILE = os.environ["AWS_PROFILE"]
REGION = "eu-west-1"

# Parallel scan segments — increase for larger tables
TOTAL_SEGMENTS = 16

# Max items per DynamoDB BatchWriteItem / BatchDeleteItem call
BATCH_SIZE = 25

# Retry config
MAX_RETRIES = 8
BASE_BACKOFF = 0.1  # seconds

boto_config = Config(
    retries={"max_attempts": 10, "mode": "adaptive"},
    max_pool_connections=TOTAL_SEGMENTS + 4,
)
session = boto3.Session(profile_name=AWS_PROFILE, region_name=REGION)

# --- Verify AWS identity ---
sts = session.client("sts")
identity = sts.get_caller_identity()
print(f"AWS Account:  {identity['Account']}")
print(f"ARN:          {identity['Arn']}")
print(f"Profile:      {AWS_PROFILE}")
print(f"Region:       {REGION}")
print()

AWS Account:  760097843905
ARN:          arn:aws:sts::760097843905:assumed-role/platform-developer/botocore-session-1773833326
Profile:      platform-developer
Region:       eu-west-1



In [2]:
CORRUPTED_TABLE = input("Table to DELETE all items from: ").strip()
RESTORED_TABLE = input("Restored table to copy FROM:   ").strip()

assert CORRUPTED_TABLE, "Corrupted table name must not be empty"
assert RESTORED_TABLE, "Restored table name must not be empty"
assert CORRUPTED_TABLE != RESTORED_TABLE, "Table names must differ"

print()
print(f"  DELETE all items from → {CORRUPTED_TABLE}")
print(f"  COPY all items from   → {RESTORED_TABLE}")
print()

confirm = input(f"Type the corrupted table name to confirm. This action will DELETE all items: ").strip()
if confirm != CORRUPTED_TABLE:
    raise RuntimeError(f"Confirmation mismatch: '{confirm}' != '{CORRUPTED_TABLE}'. Aborting.")

print("✅ Confirmed. Proceeding.")


  DELETE all items from → test-restore-script-deleteme
  COPY all items from   → restored-test-restore-script-deleteme

✅ Confirmed. Proceeding.


In [3]:
dynamodb = session.resource("dynamodb", config=boto_config)
dynamodb_client = session.client("dynamodb", config=boto_config)

corrupted_table = dynamodb.Table(CORRUPTED_TABLE)
restored_table = dynamodb.Table(RESTORED_TABLE)

print(f"Corrupted table: {CORRUPTED_TABLE}")
print(f"Restored table:  {RESTORED_TABLE}")
print(f"Parallel segments: {TOTAL_SEGMENTS}")

Corrupted table: test-restore-script-deleteme
Restored table:  restored-test-restore-script-deleteme
Parallel segments: 16


## Helpers

In [4]:
def exponential_backoff(attempt: int) -> None:
    """Sleep with jittered exponential backoff."""
    delay = BASE_BACKOFF * (2 ** attempt) + random.uniform(0, BASE_BACKOFF)
    time.sleep(delay)


def batch_delete_with_retries(table_name: str, keys: list[dict]) -> int:
    """Delete a batch of keys with retry on UnprocessedItems. Returns number deleted."""
    request_items = {table_name: [{"DeleteRequest": {"Key": key}} for key in keys]}
    deleted = 0

    for attempt in range(MAX_RETRIES):
        response = dynamodb_client.batch_write_item(
            RequestItems=request_items
        )
        unprocessed = response.get("UnprocessedItems", {})
        processed_count = len(keys) - len(unprocessed.get(table_name, []))
        deleted += processed_count

        if not unprocessed or not unprocessed.get(table_name):
            return deleted

        request_items = unprocessed
        keys = [
            req["DeleteRequest"]["Key"] for req in unprocessed[table_name]
        ]
        print(f"  Retrying {len(keys)} unprocessed deletes (attempt {attempt + 1})")
        exponential_backoff(attempt)

    raise RuntimeError(
        f"Still have {len(keys)} unprocessed deletes after {MAX_RETRIES} retries"
    )


def get_item_count(table_name: str) -> int:
    """Get the approximate item count from table metadata."""
    resp = dynamodb_client.describe_table(TableName=table_name)
    return resp["Table"]["ItemCount"]


def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i : i + n]


print("Helpers loaded.")

Helpers loaded.


## Pre-flight checks

Verify both tables exist and print their current item counts.

In [5]:
corrupted_count = get_item_count(CORRUPTED_TABLE)
restored_count = get_item_count(RESTORED_TABLE)

print(f"Corrupted table ({CORRUPTED_TABLE}): ~{corrupted_count:,} items")
print(f"Restored table  ({RESTORED_TABLE}): ~{restored_count:,} items")
print()
print("⚠️  The next cell will DELETE ALL items from the corrupted table.")
print("    Make sure you are certain before proceeding.")

Corrupted table (test-restore-script-deleteme): ~0 items
Restored table  (restored-test-restore-script-deleteme): ~0 items

⚠️  The next cell will DELETE ALL items from the corrupted table.
    Make sure you are certain before proceeding.


## Step 1: Clear the corrupted table

Parallel scan + batch delete. The table key is `id` (String).

In [ ]:
def delete_segment(segment: int) -> int:
    """Scan and delete all items in one parallel-scan segment."""
    deleted = 0
    scan_kwargs = {
        "TableName": CORRUPTED_TABLE,
        "ProjectionExpression": "id",
        "TotalSegments": TOTAL_SEGMENTS,
        "Segment": segment,
    }

    while True:
        response = dynamodb_client.scan(**scan_kwargs)
        items = response.get("Items", [])

        if items:
            keys = [{"id": item["id"]} for item in items]
            for batch in chunks(keys, BATCH_SIZE):
                deleted += batch_delete_with_retries(CORRUPTED_TABLE, batch)

        if "LastEvaluatedKey" not in response:
            break
        scan_kwargs["ExclusiveStartKey"] = response["LastEvaluatedKey"]

    return deleted


def delete_all_items() -> int:
    """Repeatedly scan+delete until the table is confirmed empty."""
    grand_total = 0
    pass_num = 0

    while True:
        pass_num += 1
        print(f"\n--- Delete pass {pass_num} ---")

        with ThreadPoolExecutor(max_workers=TOTAL_SEGMENTS) as executor:
            futures = {
                executor.submit(delete_segment, seg): seg
                for seg in range(TOTAL_SEGMENTS)
            }
            pass_deleted = 0
            for future in as_completed(futures):
                seg = futures[future]
                count = future.result()
                pass_deleted += count
                if count > 0:
                    print(f"  Segment {seg:2d}: deleted {count:,} items")

        grand_total += pass_deleted
        print(f"  Pass {pass_num} deleted {pass_deleted:,} items (total so far: {grand_total:,})")

        if pass_deleted == 0:
            break

    # Final verification: confirm the table is truly empty
    print("\nVerifying table is empty...")
    verify_kwargs = {
        "TableName": CORRUPTED_TABLE,
        "Select": "COUNT",
        "ConsistentRead": True,
        "Limit": 1,
    }
    resp = dynamodb_client.scan(**verify_kwargs)
    remaining = resp["Count"]
    if remaining == 0:
        print("✅ Table is confirmed empty.")
    else:
        raise RuntimeError(
            f"❌ Table still has items after {pass_num} passes! "
            f"Sample scan returned {remaining} item(s). Re-run this cell."
        )

    return grand_total


print(f"Deleting all items from {CORRUPTED_TABLE} using {TOTAL_SEGMENTS} parallel segments...")
start = time.time()
total_deleted = delete_all_items()
elapsed = time.time() - start
print(f"\nDone. Deleted {total_deleted:,} items in {elapsed:.1f}s")

Deleting all items from test-restore-script-deleteme using 16 parallel segments...

--- Delete pass 1 ---
  Pass 1 deleted 0 items (total so far: 0)

Verifying table is empty...
✅ Table is confirmed empty.

Done. Deleted 0 items in 0.2s


## Step 2: Copy items from restored table to original table

Parallel scan of the restored table + batch write into the original table.

In [7]:
def copy_segment(segment: int) -> int:
    """Scan one segment of the restored table and write to the corrupted table."""
    copied = 0
    # Use the high-level Table resource for scan so items come back as
    # native Python types (the low-level client returns DynamoDB typed dicts
    # which would need to be converted for batch_write_item).
    scan_kwargs = {
        "TotalSegments": TOTAL_SEGMENTS,
        "Segment": segment,
    }

    # Single batch_writer per segment — flushes automatically every 25 items
    # and on context-manager exit, avoiding per-batch open/close overhead.
    with corrupted_table.batch_writer() as writer:
        while True:
            response = restored_table.scan(**scan_kwargs)
            items = response.get("Items", [])

            for item in items:
                writer.put_item(Item=item)
            copied += len(items)

            if "LastEvaluatedKey" not in response:
                break
            scan_kwargs["ExclusiveStartKey"] = response["LastEvaluatedKey"]

    return copied


print(f"Copying items from {RESTORED_TABLE} → {CORRUPTED_TABLE}...")
print(f"Using {TOTAL_SEGMENTS} parallel segments.")
start = time.time()
total_copied = 0

with ThreadPoolExecutor(max_workers=TOTAL_SEGMENTS) as executor:
    futures = {
        executor.submit(copy_segment, seg): seg
        for seg in range(TOTAL_SEGMENTS)
    }
    for future in as_completed(futures):
        seg = futures[future]
        count = future.result()
        total_copied += count
        print(f"  Segment {seg:2d}: copied {count:,} items")

elapsed = time.time() - start
print(f"\nDone. Copied {total_copied:,} items in {elapsed:.1f}s")

Copying items from restored-test-restore-script-deleteme → test-restore-script-deleteme...
Using 16 parallel segments.
  Segment  1: copied 0 items
  Segment  4: copied 0 items
  Segment  6: copied 0 items
  Segment  0: copied 0 items
  Segment  2: copied 0 items
  Segment  9: copied 0 items
  Segment  3: copied 0 items
  Segment 14: copied 0 items
  Segment  7: copied 0 items
  Segment 10: copied 0 items
  Segment  8: copied 0 items
  Segment 13: copied 0 items
  Segment  5: copied 0 items
  Segment 15: copied 0 items
  Segment 12: copied 0 items
  Segment 11: copied 0 items

Done. Copied 0 items in 0.1s


## Step 3: Verification

Compare item counts using strongly consistent parallel scans (2x read cost, but
gives an accurate count immediately after writes/deletes).

In [8]:
def count_items_via_scan(table_name: str) -> int:
    """Count items using a strongly consistent parallel scan."""
    def count_segment(segment: int) -> int:
        count = 0
        scan_kwargs = {
            "TableName": table_name,
            "Select": "COUNT",
            "ConsistentRead": True,
            "TotalSegments": TOTAL_SEGMENTS,
            "Segment": segment,
        }
        while True:
            resp = dynamodb_client.scan(**scan_kwargs)
            count += resp["Count"]
            if "LastEvaluatedKey" not in resp:
                break
            scan_kwargs["ExclusiveStartKey"] = resp["LastEvaluatedKey"]
        return count

    with ThreadPoolExecutor(max_workers=TOTAL_SEGMENTS) as executor:
        counts = list(executor.map(count_segment, range(TOTAL_SEGMENTS)))
    return sum(counts)


print("Counting items (strongly consistent parallel scan)...")
start = time.time()
final_corrupted = count_items_via_scan(CORRUPTED_TABLE)
final_restored = count_items_via_scan(RESTORED_TABLE)
elapsed = time.time() - start

print(f"  {CORRUPTED_TABLE}: {final_corrupted:,} items")
print(f"  {RESTORED_TABLE}:  {final_restored:,} items")
print(f"  (counted in {elapsed:.1f}s)")
print()

if final_corrupted == final_restored:
    print("✅ Item counts match. Restore looks successful.")
elif final_corrupted > final_restored:
    diff = final_corrupted - final_restored
    print(f"❌ MISMATCH: target table has {diff:,} MORE items than the restored table.")
    print("   The corrupted table may not have been fully cleared before copying.")
else:
    diff = final_restored - final_corrupted
    print(f"❌ MISMATCH: target table has {diff:,} FEWER items than the restored table.")
    print("   Some items may have failed to copy.")

Counting items (strongly consistent parallel scan)...
  test-restore-script-deleteme: 0 items
  restored-test-restore-script-deleteme:  0 items
  (counted in 0.1s)

✅ Item counts match. Restore looks successful.
